In [ ]:
import gymnasium as gym
import flappy_bird_gymnasium
import pygame

# creating our env
env = gym.make("FlappyBird-v0", render_mode="human")
state, info = env.reset()
done = False

# Initialize PyGame keyboard
pygame.init()
screen = pygame.display.get_surface()  # Gym has already created a window

while not done:
    action = 0  # default -> 0 is no flap & 1 is flap

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                action = 1 # flap

    state, reward, done, truncated, info = env.step(action)
    env.render()

env.close()
pygame.quit()

In [2]:
# Agent- agent.py

import flappy_bird_gymnasium
import gymnasium as gym
from dqn import DQN
from experience_replay import ReplayMemory
import itertools
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
import os
import argparse
import random



if torch.backends.mps.is_available():
    device="mps"
elif torch.cuda.is_available():
    device="cuda"
else
    device="cpu"
# inside google collab change runtime => t4 gpu 

RUNS_DIR = "runs"
os.makedirs(RUNS_DIR, exist_ok=True)


class Agent:
    def __init__(self, param_set):
        self.param_set = param_set
        
        with open("parameters.yaml", "r") as f:
            all_param_set = yaml.safe_load(f)
            params = all_param_set[param_set]

        self.alpha = params["alpha"]
        self.gamma = params["gamma"]
        
        self.epsilon_init = params["epsilon_init"]
        self.epsilon_min = params["epsilon_min"]
        self.epsilon_decay = params["epsilon_decay"]
        
        self.replay_memory_size = params["replay_memory_size"]
        self.mini_batch_size = params["mini_batch_size"]

        self.reward_threshold = params["reward_threshold"]
        self.network_sync_rate = params["network_sync_rate"]
        self.mini_batch_size = params["mini_batch_size"]

        self.loss_fn = nn.MSELoss()
        self.optimizer = None

        self.LOG_FILE = os.path.join(RUNS_DIR, f"{self.param_set}.log")
        self.MODEL_FILE = os.path.join(RUNS_DIR, f"{self.param_set}.pt")
    
    
    def run(self, is_training=True, render=False):

        env = gym.make("FlappyBird-v0", render_mode="human" if render else None)   #use_lidar=True

        num_states = env.observation_space.shape[0]  # input dim
        num_actions = env.action_space.n  # output dim

        policy_dqn = DQN(num_states, num_actions).to(device)


        if is_training:
            memory = ReplayMemory(self.replay_memory_size)
            epsilon = self.epsilon_init

            target_dqn = DQN(num_states, num_actions).to(device)
            # copy the wt & bias vals from policy => target
            target_dqn.load_state_dict(policy_dqn.state_dict())

            steps = 0

            self.optimizer = optim.Adam(policy_dqn.parameters(), lr=self.alpha)

            best_reward = float("-inf")

        else:
            # best policy load
            policy_dqn.load_state_dict(torch.load(self.MODEL_FILE))
            policy_dqn.eval()
            

        for episode in itertools.count():
            state, _ = env.reset()
            state = torch.tensor(state, dtype=torch.float, device=device)
            
            episode_reward = 0
            terminated = False
        
            while (not terminated and episode_reward < self.reward_threshold):
                if is_training and random.random() < epsilon:
                    action = env.action_space.sample() # explore
                    action = torch.tensor(action, dtype=torch.long, device=device)
                else:
                    with torch.no_grad():
                        action = policy_dqn(state.unsqueeze(dim=0)).squeeze().argmax() # exploit
                
                # Next action:
                # (feed the observation to your agent here)
                # action = env.action_space.sample()

                # Processing: terminated => done
                next_state, reward, terminated, _, _ = env.step(action.item())

                episode_reward += reward

                # create tensors
                reward = torch.tensor(reward, dtype=torch.float, device=device)
                next_state = torch.tensor(next_state, dtype=torch.float, device=device)

                if is_training:
                    memory.append((state, action, next_state, reward,  terminated))
                    steps += 1

                state = next_state
                

            print(f"for episode={episode+1} with total reward={episode_reward} & epsilon={epsilon}")


            if is_training:
                # epsilon decay
                epsilon = max(epsilon * self.epsilon_decay, self.epsilon_min)

                if episode_reward > best_reward:
                    log_msg = f"best reward = {episode_reward} for episode={episode+1}"

                    with open(self.LOG_FILE, "a") as f:
                        f.write(log_msg + "\n")

                    torch.save(policy_dqn.state_dict(), self.MODEL_FILE)
                    best_reward = episode_reward
                

            if is_training and len(memory) > self.mini_batch_size:
                # get sample
                mini_batch = memory.sample(self.mini_batch_size)
                
                self.optimize(mini_batch, policy_dqn, target_dqn)

                # sync the network
                if steps > self.network_sync_rate:
                    target_dqn.load_state_dict(policy_dqn.state_dict())
                    steps = 0

    
    def optimize(self, mini_batch, policy_dqn, target_dqn):
        # get batch of experiences
        states, action, next_states, reward, terminations = zip(*mini_batch)

        states = torch.stack(states)
        actions = torch.stack(actions)
        next_states = torch.stack(next_states)
        rewards = torch.stack(rewards)
        terminations = torch.tensor(terminations).float().to(device)

        # calculate target Q-values - if terminations=true => zero
        with torch.no_grad():
            target_q = rewards + (1-terminations) * self.gamma * target_dqn(next_states).max(dim=1)[0]

        # calculate y_pred i.e Q-value from current policy
        current_q = policy_dqn(states).gather(dim=1, index=actions.unsqueeze(dim=1)).squeeze()

        # compute loss
        loss = self.loss_fn(current_q, target_q)

        # optimize model
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


if __name__ =="__main__":
    # parse command line inputs
    parser = argparse.ArgumentParser(description='Train or test model.')
    parser.add_argument('hyperparameters', help='')
    parser.add_argument('--train', help='Training mode', action='store_true')
    args = parser.parse_args()

    dql = Agent(param_set=args.hyperparameters)

    if args.train:
        dql.run(is_training=True)
    else:
        dql.run(is_training=False, render=True)
        

# -----------------------------------------------------------------------------------------
   
        # for state, action, next_state, reward, terminated in mini_batch:

        #     if terminated:
        #         target = reward
        #     else:
        #         with torch.no_grad():
        #             target_q = reward + self.gamma * target_dqn(next_state).max()

        #     current_q = policy_dqn(state)


        #     #loss
        #     loss = self.loss_fn(current_q, target_q)

        #     self.optimizer.zero_grad()
        #     loss.backward()
        #     self.optimizer.step()
            

            # Checking if the player is still alive
            # if terminated:
            #     break
    
        #env.close()  --- manually stop

# -----------------------------------------------------------------------------------------
    

# def run(self, is_training=True, render=False):

#     env = gym.make("FlappyBird-v0", render_mode="human" if render else None)   #use_lidar=True

#     num_states = env.observation_space.shape[0]  # input dim
#     num_actions = env.action_space.n  # output dim

#     policy_dqn = DQN(num_states, num_actions).to(device)


#     if is_training:
#         memory = ReplayMemory(10000)

#     for episode in itertools.count():
#         state, _ = env.reset()
#         episode_rewards = 0
#         terminated = False
        
#         while not terminated:
#             # Next action:
#             # (feed the observation to your agent here)
#             action = env.action_space.sample()

#             # Processing: terminated => done
#             next_state, reward, terminated, _, _ = env.step(action)

#             if is_training:
#                 memory.append((state, action, new_state, reward,  terminated))

#             state = new_state
#             episode_rewards += rewards

#         print(f"for episode={episode+1} with total reward={episode_rewards}")

            # Checking if the player is still alive
            # if terminated:
            #     break
    
        #env.close()  --- manually stop     

ModuleNotFoundError: No module named 'flappy_bird_gymnasium'

In [ ]:
# DQN

import torch
import torch.nn as nn

class DQN(nn.Module):
    # input dim, output dim, hidden dim
    def __init__(self, state_dim=12, action_dim=2, hidden_dim=256):
        super(DQN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.model(x)
        

In [ ]:
# experience_replay.py

from collection import deque
import random

class ReplayMemory():
    
    # create FIFO queue - experience replay
    def __init__(self, maxlen, seed=None):
        self.memory = deque([], maxlen=maxlen)

    def append(self, new_exp):
        self.memory.append(new_exp)

    def sample(self, sample_size):
        return random.sample(self.memory, sample_size)

    # curr buffer size
    def __len__(self):
        return len(self.memory)

In [ ]:
# parameters.yaml

flappybirdv0:
    env_id: FlappyBird-v0
    epsilon_init: 1
    epsilon_min: 0.05
    epsilon_decay: 0.9995
    replay_memory_size: 100000
    mini_batch_size: 32
    network_sync_rate= 10
    alpha: 0.001
    gamma: 0.99
    reward_threshold: 1000